# 01 Preprocesamiento

Este notebook prepara los datos base del problema de optimizacion combinatoria sobre las capitales departamentales de Francia.

Objetivos de esta etapa:
- cargar el dataset original de rutas directas
- revisar la estructura de las variables disponibles
- documentar la funcion de costo usada en el proyecto
- construir las aristas ponderadas del grafo
- exportar los artefactos reutilizables a `data/processed`


In [19]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

ROOT

WindowsPath('C:/Carlos/Uni/Algortimos/trabajos/trabajo_1/trabajo_carlos/2. optimizacion_combinatoria')

In [20]:
import pandas as pd

from src.config_modelo_costo import SMIC_HOURLY_EUR_2026, VEHICLE, cost_formula_label, model_metadata
from src.core_paths import PHASE1_DIR, RAW_CONNECTIONS_DATASET_PATH
from src.core_preprocesamiento import run_preprocessing


## Dataset original

El archivo fuente contiene conexiones directas entre pares de ciudades, junto con distancia, tiempo, peajes y costo estimado de gasolina.

In [21]:
df_raw = pd.read_csv(RAW_CONNECTIONS_DATASET_PATH)
print('Ruta del dataset:', RAW_CONNECTIONS_DATASET_PATH)
print('Shape:', df_raw.shape)
df_raw.head()

Ruta del dataset: C:\Carlos\Uni\Algortimos\trabajos\trabajo_1\trabajo_carlos\2. optimizacion_combinatoria\data\raw\dataset_cities_france.csv
Shape: (242, 9)


,indice_inicio,indice_destino,ciudad_inicio,ciudad_destino,distancia,tiempo(min),peajes(euros),gasolina(euros),Unnamed: 8
0,0,39,Bourg-en-Bresse,Lons-le-Saunier,74,59,5.3,15.34,NaN
1,0,69,Bourg-en-Bresse,Lyon,84,69,9.3,17.94,NaN
2,0,71,Bourg-en-Bresse,Mâcon,84,69,9.3,17.94,NaN
3,0,74,Bourg-en-Bresse,Annecy,84,69,9.3,17.94,NaN
4,1,7,Laon,Charleville-Mézières,112,102,0.0,26.52,NaN


In [22]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 242 entries, 0 to 241
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   indice_inicio    242 non-null    int64  
 1   indice_destino   242 non-null    int64  
 2   ciudad_inicio    242 non-null    str    
 3   ciudad_destino   242 non-null    str    
 4   distancia        242 non-null    int64  
 5   tiempo(min)      242 non-null    int64  
 6   peajes(euros)    242 non-null    float64
 7   gasolina(euros)  242 non-null    float64
 8   Unnamed: 8       0 non-null      float64
dtypes: float64(3), int64(4), str(2)
memory usage: 17.1 KB


## Funcion de costo

La funcion de costo alineada con el proyecto es:

`peajes(euros) + gasolina(euros) + (tiempo(min) / 60) * tarifa_vendedor`

Se usa como referencia un vendedor en carro con salario horario igual al SMIC de Francia 2026.

In [23]:
metadata = model_metadata()
print('Formula:', cost_formula_label())
print('Tarifa vendedor EUR/h:', SMIC_HOURLY_EUR_2026)
print('Vehiculo de referencia:', VEHICLE)
metadata

Formula: peajes(euros) + gasolina(euros) + (tiempo(min)/60) * 12.02
Tarifa vendedor EUR/h: 12.02
Vehiculo de referencia: VehicleSpec(name='Renault Clio', fuel_type='gasolina', segment='compacto', note='Vehiculo de referencia para justificar el recorrido. El componente de combustible se toma desde la columna gasolina(euros) del dataset.')


{'vehiculo_referencia': {'name': 'Renault Clio',
  'fuel_type': 'gasolina',
  'segment': 'compacto',
  'note': 'Vehiculo de referencia para justificar el recorrido. El componente de combustible se toma desde la columna gasolina(euros) del dataset.'},
 'tarifa_vendedor_eur_h': 12.02,
 'salario_referencia': 'SMIC Francia 2026',
 'formula_peso': 'peajes(euros) + gasolina(euros) + (tiempo(min)/60) * 12.02'}

## Construccion de aristas ponderadas

Esta celda ejecuta el preprocesamiento reusable desde `src/` y deja la salida principal del paso:
- `aristas_ponderadas.csv`


In [24]:
pre = run_preprocessing()
pre['summary']

{'ciudades': 96,
 'conexiones_directas': 242,
 'peso_minimo': 11.968667,
 'peso_maximo': 346.074667,
 'peso_promedio': 50.013442,
 'formula_peso': 'peajes(euros) + gasolina(euros) + (tiempo(min)/60) * 12.02',
 'vehiculo_referencia': {'name': 'Renault Clio',
  'fuel_type': 'gasolina',
  'segment': 'compacto',
  'note': 'Vehiculo de referencia para justificar el recorrido. El componente de combustible se toma desde la columna gasolina(euros) del dataset.'},
 'tarifa_vendedor_eur_h': 12.02,
 'salario_referencia': 'SMIC Francia 2026'}

In [25]:
df_edges = pd.DataFrame(pre['edges'])
print('Salida principal:', pre['weighted_edges_path'])
df_edges.head()

Salida principal: C:\Carlos\Uni\Algortimos\trabajos\trabajo_1\trabajo_carlos\2. optimizacion_combinatoria\data\processed\aristas_ponderadas.csv


,indice_inicio,indice_destino,ciudad_inicio,ciudad_destino,peso,distancia,tiempo(min),peajes(euros),gasolina(euros),costo_vendedor(euros)
0,0,39,Bourg-en-Bresse,Lons-le-Saunier,32.459667,74.0,59.0,5.3,15.34,11.819667
1,0,69,Bourg-en-Bresse,Lyon,41.063000,84.0,69.0,9.3,17.94,13.823000
2,0,71,Bourg-en-Bresse,Mâcon,41.063000,84.0,69.0,9.3,17.94,13.823000
3,0,74,Bourg-en-Bresse,Annecy,41.063000,84.0,69.0,9.3,17.94,13.823000
4,1,7,Laon,Charleville-Mézières,46.954000,112.0,102.0,0.0,26.52,20.434000


In [26]:
df_edges[['peso', 'distancia', 'tiempo(min)', 'peajes(euros)', 'gasolina(euros)', 'costo_vendedor(euros)']].describe().T

,count,mean,std,min,25%,50%,75%,max
peso,242.0,50.013442,34.706098,11.968667,32.837583,44.8825,57.517833,346.074667
distancia,242.0,122.194215,77.218069,13.000000,80.250000,114.0000,150.000000,696.000000
tiempo(min),242.0,96.855372,68.782354,25.000000,65.250000,87.0000,110.750000,704.000000
peajes(euros),242.0,5.427686,7.088330,0.000000,0.000000,2.6000,9.300000,43.700000
gasolina(euros),242.0,25.182397,17.883412,6.500000,16.965000,22.6200,28.795000,183.040000
costo_vendedor(euros),242.0,19.403360,13.779398,5.008333,13.071750,17.4290,22.186917,141.034667


## Validaciones rapidas

Chequeamos el numero de ciudades, conexiones directas y que el archivo procesado exista donde lo espera el resto del pipeline.

In [27]:
print('Directorio de salida:', PHASE1_DIR)
print('Existe aristas_ponderadas.csv:', pre['weighted_edges_path'].exists())
print('Numero de ciudades:', len(pre['cities']))
print('Numero de conexiones directas:', len(pre['edges']))

Directorio de salida: C:\Carlos\Uni\Algortimos\trabajos\trabajo_1\trabajo_carlos\2. optimizacion_combinatoria\data\processed
Existe aristas_ponderadas.csv: True
Numero de ciudades: 96
Numero de conexiones directas: 242


## Salida de esta etapa

Con esto queda lista la base para el notebook `02_optimizacion.ipynb`, que construira las matrices completas y ejecutara los algoritmos ACO y GA sobre el problema ya preprocesado.